-------------------------------------------------------------------------
*   PONTIFÍCIA UNIVERSIDADE CATÓLICA DE MINAS GERAIS
*   PROFESSOR: VICTOR SALES SILVA
*   ALUNO: DGEISON SERRÃO PEIXOTO
*   MATRÍCULA: **1366415**
*   ATIVIDADE: LEITURA DE ARQUIVO EM FORMATO XML UTILIZANDO SPARK
-------------------------------------------------------------------------

# INSTALAÇÃO DAS BIBLIOTECAS

In [ ]:
%pip install pyspark
!pip install azure-storage-blob

# IMPORTAÇÃO DAS BIBLIOTECAS

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_date
import xml.etree.ElementTree as ET
from azure.storage.blob import BlobServiceClient
from azure.storage.blob import BlobClient

# CRIAÇÃO DA APLICAÇÃO SPARK

In [ ]:
spark = SparkSession.builder.getOrCreate()

# VARIÁVEIS DE APOIO

In [ ]:
storageaccount = 'stgaccount687878'
container = 'datalake-687878'
connection_string = 'DefaultEndpointsProtocol=https;AccountName=stgaccount687878;AccountKey=SUA_ACCOUNT_KEY_AQUI;EndpointSuffix=core.windows.net'
blob_file = 'bronze/DADOS_ALUNOS/DADOS_ALUNOS.xml'

In [ ]:
def listar_arquivos_no_container(conn_string, container_name, prefixo=""):
  try:
      # 1. Conecta ao serviço de Blob
      blob_service_client = BlobServiceClient.from_connection_string(conn_string)

      # 2. Obtém o cliente para o contêiner
      container_client = blob_service_client.get_container_client(container_name)

      print(f"Buscando arquivos em '{container_name}' com o prefixo '{prefixo}'...")

      # 3. Lista os blobs (arquivos) que começam com o prefixo
      blob_list = container_client.list_blobs(name_starts_with=prefixo)

      lista_de_arquivos = []
      for blob in blob_list:
          print(f"  - {blob.name}")
          lista_de_arquivos.append(blob.name)

      if not lista_de_arquivos:
          print("\nNenhum arquivo encontrado neste caminho.")

      return lista_de_arquivos

  except Exception as e:
      print(f"Ocorreu um erro ao tentar listar os arquivos: {e}")
      return []


camada_para_verificar = 'bronze/'

print(f"--- Verificando o conteúdo da camada '{camada_para_verificar}' ---")
arquivos_encontrados = listar_arquivos_no_container(
    connection_string,
    container,
    prefixo=camada_para_verificar
)

print("\n--- Fim da verificação ---")

--- Verificando o conteúdo da camada 'bronze/' ---
Buscando arquivos em 'datalake-687878' com o prefixo 'bronze/'...
  - bronze/DADOS_ALUNOS/DADOS_ALUNOS.xml
  - bronze/DADOS_BANCARIOS/DADOS_BANCARIOS.xml
  - bronze/DADOS_ESTUDANTES/DADOS_ESTUDANTES.json
  - bronze/DADOS_EXAMES/DADOS_EXAMES.csv
  - bronze/DADOS_VOOS/DADOS_VOOS.parquet

--- Fim da verificação ---


# FUNÇÃO PARA LER ARQUIVO XML

In [ ]:
blob_file_na_nuvem = 'bronze/DADOS_ALUNOS/DADOS_ALUNOS.xml'
arquivo_local = 'DADOS_ALUNOS.xml'

print(f"Baixando o arquivo '{blob_file_na_nuvem}' da nuvem...")
try:
    blob_client = BlobClient.from_connection_string(
        conn_str=connection_string,
        container_name=container,
        blob_name=blob_file_na_nuvem
    )
    with open(arquivo_local, "wb") as my_blob:
        blob_data = blob_client.download_blob()
        blob_data.readinto(my_blob)

    print(f"Arquivo salvo localmente como '{arquivo_local}' com sucesso!")

except Exception as e:
    print(f"Ocorreu um erro no download: {e}")
    arquivo_local = None


Baixando o arquivo 'bronze/DADOS_ALUNOS/DADOS_ALUNOS.xml' da nuvem...
Arquivo salvo localmente como 'DADOS_ALUNOS.xml' com sucesso!


In [ ]:
def ler_xml(caminho_do_arquivo):
  try:
      tree = ET.parse(caminho_do_arquivo)
      root = tree.getroot()
      return root
  except FileNotFoundError:
      print(f"Erro: O arquivo '{caminho_do_arquivo}' não foi encontrado.")
      return None
  except ET.ParseError:
      print(f"Erro: O arquivo '{caminho_do_arquivo}' não é um XML válido.")
      return None


In [ ]:
if arquivo_local:
    print("\nLendo o arquivo XML que foi baixado...")
    root = ler_xml(arquivo_local)
    if root is not None:
        print("\nLeitura do XML bem-sucedida!")
        print(f"Elemento raiz: <{root.tag}>")
        print("Alguns elementos filhos:")
        count = 0
        for child in root:
            if count < 5:
                print(f"  - <{child.tag}>")
                count += 1
            else:
                break


Lendo o arquivo XML que foi baixado...

Leitura do XML bem-sucedida!
Elemento raiz: <alunos>
Alguns elementos filhos:
  - <aluno>
  - <aluno>
  - <aluno>
  - <aluno>
  - <aluno>


# CRIAÇÃO DE LISTA COM O CONTEÚDO DO XML

In [ ]:
root = ler_xml(arquivo)
dados_alunos = []

for aluno in root:
  nome_completo = aluno.find('nome_completo').text
  data_nascimento = aluno.find('data_nascimento').text
  bairro = aluno.find('bairro').text
  cidade = aluno.find('cidade').text
  estado = aluno.find('estado').text
  tipo_escola = aluno.find('tipo_escola').text
  nome_escola = aluno.find('nome_escola').text
  for notas in aluno:
    for disciplinas in notas:
      disciplina = str(disciplinas.attrib.get('nome'))
      for atividades in disciplinas:
        atividade = atividades.find('nome').text
        valor = atividades.find('valor').text
        nota = atividades.find('nota').text
        dados_alunos.append([nome_completo, data_nascimento, bairro, cidade, estado, tipo_escola, nome_escola, disciplina, atividade, valor, nota])

# LEITURA DA LISTA USANDO SPARK

In [ ]:
df = spark.createDataFrame(dados_alunos, ['nome_completo', 'data_nascimento', 'bairro', 'cidade', 'estado', 'tipo_escola', 'nome_escola', 'disciplina', 'atividade', 'valor', 'nota'])

# EXIBINDO UMA AMOSTRA DOS DADOS

In [ ]:
df.show(truncate=False)

+----------------------+---------------+-----------+--------------+------------+-----------+-----------------------+----------+-----------+-----+----+
|nome_completo         |data_nascimento|bairro     |cidade        |estado      |tipo_escola|nome_escola            |disciplina|atividade  |valor|nota|
+----------------------+---------------+-----------+--------------+------------+-----------+-----------------------+----------+-----------+-----+----+
|Luiz Miguel Nascimento|09/02/2009     |Mangabeiras|Belo Horizonte|Minas Gerais|Municipal  |Escola Estadual Almeida|Português |Atividade 1|56   |52  |
|Luiz Miguel Nascimento|09/02/2009     |Mangabeiras|Belo Horizonte|Minas Gerais|Municipal  |Escola Estadual Almeida|Português |Atividade 2|31   |14  |
|Luiz Miguel Nascimento|09/02/2009     |Mangabeiras|Belo Horizonte|Minas Gerais|Municipal  |Escola Estadual Almeida|Português |Atividade 3|6    |1   |
|Luiz Miguel Nascimento|09/02/2009     |Mangabeiras|Belo Horizonte|Minas Gerais|Municipal  |Es

# EXIBINDO OS METADADOS (SCHEMA) DO ARQUIVO

In [ ]:
df.printSchema()

root
 |-- nome_completo: string (nullable = true)
 |-- data_nascimento: string (nullable = true)
 |-- bairro: string (nullable = true)
 |-- cidade: string (nullable = true)
 |-- estado: string (nullable = true)
 |-- tipo_escola: string (nullable = true)
 |-- nome_escola: string (nullable = true)
 |-- disciplina: string (nullable = true)
 |-- atividade: string (nullable = true)
 |-- valor: string (nullable = true)
 |-- nota: string (nullable = true)



# AJUSTANDO O SCHEMA DOS DADOS, SE NECESSÁRIO

In [ ]:
df = df.withColumn('nota', df['nota'].cast('double'))
df = df.withColumn('valor', df['valor'].cast('double'))
df = df.withColumn("data_nascimento", to_date("data_nascimento", "dd/MM/yyyy"))
df = df.withColumn('data_nascimento', df['data_nascimento'].cast('date'))

In [ ]:
df.show(truncate=False)

+----------------------+---------------+-----------+--------------+------------+-----------+-----------------------+----------+-----------+-----+----+
|nome_completo         |data_nascimento|bairro     |cidade        |estado      |tipo_escola|nome_escola            |disciplina|atividade  |valor|nota|
+----------------------+---------------+-----------+--------------+------------+-----------+-----------------------+----------+-----------+-----+----+
|Luiz Miguel Nascimento|2009-02-09     |Mangabeiras|Belo Horizonte|Minas Gerais|Municipal  |Escola Estadual Almeida|Português |Atividade 1|56.0 |52.0|
|Luiz Miguel Nascimento|2009-02-09     |Mangabeiras|Belo Horizonte|Minas Gerais|Municipal  |Escola Estadual Almeida|Português |Atividade 2|31.0 |14.0|
|Luiz Miguel Nascimento|2009-02-09     |Mangabeiras|Belo Horizonte|Minas Gerais|Municipal  |Escola Estadual Almeida|Português |Atividade 3|6.0  |1.0 |
|Luiz Miguel Nascimento|2009-02-09     |Mangabeiras|Belo Horizonte|Minas Gerais|Municipal  |Es